# Karin routing — LoRA SFT + DPO on Colab (Drive-backed)

Fine-tunes `mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated` for tool routing. Production runs the adapter on the Jetson via Ollama + Modelfile `ADAPTER` on top of the mannix abliteration. See section 9 for deploy.

**Current production (as of 2026-04-24):** iter-3 LoRA (`run_0ac17bc7`, `karin-tuned:latest`) + the runtime layer in the Karin bridge (Phase-0 classifier patches, under-fire rescue, continuation rescue, two-phase compose, hint-in-user-msg, L8 reply scrubs). **93.3% routing / 91.9% reply / 59.2% tool-output-usage** on the 135-case held-out eval.

**Training is currently parked.** Four consecutive retrains after iter-3 (iter-4, 5, 6, 7) regressed on the 135-case eval. None shipped. Runtime-layer improvements kept working in parallel. See `docs/iter4-postmortem-*.md` through `docs/iter7-postmortem-2026-04-23.md` for per-iteration analysis and preconditions before any next attempt.

**Run identity is settings-based.** Cell 0b's `CONFIG` dict hashes to a folder name under `MyDrive/Karin_SFT/runs/`. Change a value → new hash → fresh folder. Re-run with the same CONFIG → existing folder is reused and every cell checks a `STATE` dict to skip whatever's already done. Flip `FORCE_REWRITE=True` to wipe the current run folder (the HF base-model cache and dataset tar are kept).

**Dataset hygiene is enforced.** `sft/scripts/build_dataset.py` flattens assistant `tool_calls` into `content` as plain JSON (so mlabonne's chat template doesn't drop them) and asserts zero verbatim overlap between `sft/phrase_library/train/` and `sft/eval_cases_novel.yaml` (135 held-out cases).

**All inputs and outputs live in Google Drive:**
```
MyDrive/
└── Karin_SFT/
    ├── input/
    │   └── sft_dataset.tar                 (upload once per iteration; ~4 MB)
    ├── hf_cache/                           (~16 GB base model, cached)
    └── runs/
        └── run_<hash>/                     (auto-created per CONFIG)
            ├── run_config.json             settings snapshot + hash
            ├── config.json                 build metadata
            ├── adapter/                    picker-promoted SFT adapter
            ├── adapter_dpo/                (if cell 6 ran)
            ├── checkpoint_picker.json      per-checkpoint routing scores
            └── karin-lora.gguf             ~40 MB deploy artifact
```

**Default pipeline** (run-all): SFT → checkpoint picker → DPO → LoRA GGUF → `runtime.unassign`. The merged-model path (cell 8, ~5 GB) is **OFF by default** because the LoRA GGUF from cell 8b covers the same deploy via Ollama's `ADAPTER` directive. Flip `CONFIG['run_merge_gguf']=True` if you need a standalone merged GGUF.

**Runtime target:** A100 (40 GB) completes in ~20-25 min including the picker. L4/T4 still work but expect 1.5-3× slower.

## CONFIG hyperparameters (iter-3 anti-overfit — retained)

Iter-3's settings beat the mannix baseline; they held up across iter-4/5/6/7's failed runs (each regressed on data-shape or DPO-distribution bugs, not HPs). Keep them as-is for any iter-8.

| Key | Value | Why |
|-----|-------|-----|
| `sft_epochs` | **2** | Less memorization time |
| `sft_lr` | **1e-4** | Slower descent |
| `lora_r` | **8** | 21 M trainable params (half of iter-1) |
| `lora_dropout` | **0.1** | Direct regularization |
| `weight_decay` | **0.01** | L2 keeps adapter near zero-init |

**Other config** (unchanged):
- `max_seq_length=3072` — headroom for the ~2100-token system prompt
- batch=1 × grad_accum=16 → effective batch 16
- Cosine LR + 10% eval split + early stopping (patience 3)

## Iteration history

| Iter | Run hash | Train size | Eval | Outcome |
|------|----------|-----------|------|---------|
| 1 | run_d94ae3e5 | 294 SFT + 40 DPO | 38-case: 60.5% | dropped tool_calls in training (broken) |
| 2 | run_47b09b19 | same, flatten fix | 38-case: 71.1% | fixed tool_calls, still overfit |
| 3 | run_0ac17bc7 | same, anti-overfit HPs | 38-case: 89.5% / 135-case raw: 71.1% / **with runtime stack: 93.3%** | **current prod** — `karin-tuned:latest` |
| 4 | run_52227b4c | 374 SFT + 80 DPO | 74.1% routing / 48.2% tool-output | ROLLED BACK — DPO single-turn flatten |
| 5 | run_7db68f92 | 434 SFT, DPO off | 78.5% | regressed; no-tool rebalance imbalance |
| 6 | run_d6ae0966 | 463 SFT, DPO off | 78.5% | paired positives over-generalized to decoys |
| 7 | run_f7950a38 | 463 SFT + 30 targeted DPO | 87.4% / 61.5% reply / 11.0% tool-output | ROLLED BACK — DPO reject-surface collision |

## Preconditions for iter-8

Don't retry with "a few more pairs" — three iterations of small-data targeted nudges all regressed with different failure modes (under-fire, over-fire, then schema leaks). Before a next attempt:

1. **Data budget ≥ 2-3× current** — ~450 SFT rows and ~110 DPO pairs is the plateau. Scale up or don't retry.
2. **Audited chosen-side DPO surface features** — chosen and rejected must differ only in the variable you want to change, NOT in confidence level, verb tense, or action vocabulary in general. Iter-7 failed because the reject class shared surface features with legitimate tool-grounded replies.
3. **Persist DPO trainer state** — cell 23 currently does NOT call `trainer.save_state()`. Add it, or DPO reward curves are unobservable and degenerate training is undetectable.
4. **135-case eval on Jetson** — the notebook's held-out holdout scored 80-88% on iter-5/6 and still regressed on the real 135-case eval. Only the Jetson eval counts. See `memory/feedback_eval_read_replies.md`.

Full per-iteration analysis: `docs/iter4-postmortem-*.md`, `docs/iter5-postmortem-2026-04-22.md`, `docs/iter6-postmortem-2026-04-22.md`, `docs/iter7-postmortem-2026-04-23.md`.

**Intervening runtime layer.** Every regression had a runtime fix path: classifier patches (L2a regex), under-fire rescue (L6), two-phase compose (L7), reply scrubs (L8). The runtime stack is the working strategy — each case it flips is one iter-N SFT doesn't have to address. See `docs/routing-pipeline.md` for the full layer breakdown.


## 0. Mount Google Drive + check inputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, datetime, json, sys, shutil
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/Karin_SFT')
INPUT_DIR = BASE_DIR / 'input'
RUNS_DIR = BASE_DIR / 'runs'
DATASET_TAR = INPUT_DIR / 'sft_dataset.tar'

# Create the directory structure on first run
INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_TAR.exists():
    print('=' * 70)
    print('❌ Dataset not found in Drive.')
    print('=' * 70)
    print()
    print('Upload sft/sft_dataset.tar from your local repo to:')
    print(f'  {DATASET_TAR}')
    print()
    print('Easiest way: drag the file into the Drive web UI.')
    print('Then re-run this cell.')
    raise SystemExit('Waiting for dataset upload.')

# Reduce CUDA allocator fragmentation — helps SFT eval on long sequences.
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

# ──────────────────────────────────────────────────────────────────────
# Persistent HuggingFace cache on Drive.
#
# Why: Colab runtimes are ephemeral — every fresh session re-downloads
# the ~16 GB base model (safetensors + tokenizer), costing 3-5 min
# and burning HF bandwidth. Pointing HF_HOME at Drive keeps the
# download across sessions.
#
# Storage cost: ~16 GB for Llama 3.1 8B. If you're on free-tier Drive
# (15 GB quota), this will consume most of it — flip to False if that
# matters. Colab Pro / Drive paid plans have plenty of room.
#
# First run with this ON still downloads (to Drive instead of /root).
# Every subsequent session reuses the Drive copy — near-zero download
# time, just ~30-60 s to stream 16 GB into VRAM.
# ──────────────────────────────────────────────────────────────────────
CACHE_HF_ON_DRIVE = True   # default: True — flip to False to skip caching

if CACHE_HF_ON_DRIVE:
    HF_CACHE_DIR = BASE_DIR / 'hf_cache'
    HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(HF_CACHE_DIR)
    # HF_HUB_CACHE is the new canonical var (hub >=0.20); HF_HOME sets
    # both for backwards compatibility. Also set XDG_CACHE_HOME so any
    # library consulting that falls through to Drive.
    os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
    print(f'📦 HF cache on Drive: {HF_CACHE_DIR}')
    # Quick probe — do we already have any cached models?
    hub_dir = HF_CACHE_DIR / 'hub'
    if hub_dir.exists():
        cached = list(hub_dir.glob('models--*'))
        if cached:
            print(f'   Existing models cached: {len(cached)}')
            for m in cached[:5]:
                print(f'   - {m.name}')
else:
    print('📦 HF cache: ephemeral Colab (will re-download each session)')

print(f'✅ Dataset found: {DATASET_TAR}')
os.environ['KARIN_DATASET_TAR'] = str(DATASET_TAR)

## 0b. Run identity + resume state

`CONFIG` below defines everything that makes this run unique. Its hash becomes the folder name in `MyDrive/Karin_SFT/runs/`, so **identical settings reuse the same folder** — each subsequent cell checks `STATE` and skips itself if its output already exists on Drive.

Change any value in `CONFIG` → new hash → fresh folder. Set `FORCE_REWRITE = True` to wipe the current run's artifacts (dataset + HF cache are untouched).

`RUN_NAME` is an optional human-readable tag prepended to the hash (e.g. `run_seq3072_a1b2c3d4`) — nice for skimming the `runs/` dir later.


In [ ]:
import hashlib

BASE_MODEL = 'mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated'

# ======================================================================
# Training is currently PARKED. iter-3 LoRA (run_0ac17bc7) is production.
# Four consecutive retrains (iter-4 through iter-7) regressed vs iter-3
# plus the runtime layer. Before starting iter-8, read the preconditions
# in docs/iter7-postmortem-2026-04-23.md and the iter-5/6 post-mortems.
# In short:
#   - data budget >= 2-3x current (~450 SFT rows, ~110 DPO pairs)
#   - audited chosen-side DPO surface features (differ from rejected only
#     in the target variable, not in overall confidence / verb tense)
#   - cell 23 must call trainer.save_state() (it currently doesn't -- DPO
#     reward curves aren't observable, degenerate training undetectable)
#   - only the 135-case eval on the Jetson counts; notebook holdout is
#     drawn from the same distribution as the training data
#
# If you DO run: bump `dataset_version` to a NEW string, keep the LoRA
# HPs below unchanged (they held up across every failed run), and keep
# the completion-only loss mask + label-mask probe in cell 19.
# ======================================================================

CONFIG = {
    'base_model':      BASE_MODEL,
    'lora_r':          8,
    'lora_alpha':      32,
    'max_seq_length':  3072,
    'sft_epochs':      2,
    'sft_lr':          1e-4,
    'eval_fraction':   0.10,
    'lora_dropout':    0.1,   # was 0.05 — more regularization against overfit
    'weight_decay':    0.01,  # L2 keeps LoRA weights near zero-init
    'dpo_lr':          5e-6,
    'dpo_beta':        0.1,
    'grpo_num_gens':   2,
    'grpo_beta':       0.04,
    # Bumping this invalidates old broken runs (e.g. 'tool_calls_dropped'
    # = training against literal 'None' strings). Change when dataset
    # preprocessing semantics change.
    'tool_output_format': 'json_name_arguments_v1',
    # Dataset version marker — bumping this forces a fresh hash / new
    # run folder on Drive (the run name is derived from hash(CONFIG)),
    # so each dataset revision gets its own artifacts and STATE.
    # Current value is the last executed training run (iter-6). Bump
    # to a NEW value for any future iter-7 attempt — DO NOT reuse
    # 'iter5_*' or 'iter6_*' since those point at plateau-era data.
    'dataset_version': 'iter6_rebalanced',
    # Step flags — flip off to skip a stage entirely (new hash / new run).
    'run_sft':         True,
    'run_picker':      True,
    'run_dpo':         False,    # SFT+DPO iterations (iter-4 single-turn flatten, iter-7 30-pair targeted) both catastrophically regressed prod. See project_iter4_postmortem.md + project_iter7_postmortem.md. Re-enable only with ≥2-3x data, audited chosen-side surface features, and DPO trainer_state persistence in cell 23.
    'run_grpo':        False,
    'run_merge_gguf':  False,  # cell 8: merge → 5 GB GGUF. Default OFF — cell 8b produces a 150 MB LoRA GGUF that covers the same deploy via mannix+ADAPTER. Flip True only if you need a self-contained GGUF.
    'run_lora_gguf':   True,   # cell 8b: adapter → 150 MB GGUF (Option B deploy)
}

# Optional readable tag prepended to the folder name. None = hash only.
RUN_NAME = None

# ⚠️ True wipes all training artifacts in THIS run folder.
# Does NOT touch: hf_cache/ (base model) or input/ (dataset tar).
# Set True when a run needs to be redone from scratch (wipes
# adapters, GGUFs, picker report, config — but keeps HF cache
# and input dataset). Default False so existing runs resume.
FORCE_REWRITE = False

# Hash the settings to derive a stable folder name.
_hash = hashlib.blake2b(
    json.dumps(CONFIG, sort_keys=True, default=str).encode(),
    digest_size=4,
).hexdigest()

RUN_ID = f'run_{RUN_NAME}_{_hash}' if RUN_NAME else f'run_{_hash}'
RUN_DIR = RUNS_DIR / RUN_ID

if FORCE_REWRITE and RUN_DIR.exists():
    print(f'⚠️  FORCE_REWRITE: wiping {RUN_DIR}')
    shutil.rmtree(RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Snapshot settings for this run
(RUN_DIR / 'run_config.json').write_text(
    json.dumps({**CONFIG, 'settings_hash': _hash}, indent=2, default=str)
)

# STATE = what's already complete in this run folder
STATE = {
    'sft':        (RUN_DIR / 'adapter').exists(),
    'picker':     (RUN_DIR / 'checkpoint_picker.json').exists(),
    'dpo':        (RUN_DIR / 'adapter_dpo').exists(),
    'grpo':       (RUN_DIR / 'adapter_grpo').exists(),
    'merge_gguf': (RUN_DIR / 'karin-q4_k_m.gguf').exists(),
    'lora_gguf':  (RUN_DIR / 'karin-lora.gguf').exists(),
}

print(f'Run dir: {RUN_DIR.name}')
print(f'Hash:    {_hash}')
print('Status:')
for _k, _done in STATE.items():
    _on = CONFIG.get(f'run_{_k}', True)
    if _done:
        _m = 'DONE '
    elif _on:
        _m = 'TODO '
    else:
        _m = 'SKIP '
    print(f'  [{_m}] {_k:<11s} (enabled={_on})')

os.environ['KARIN_RUN_DIR'] = str(RUN_DIR)


## 0c. GPU check

Verify a GPU is attached before we spend time on pip downloads. If this fails, the rest of the notebook will crash much later (cell 4, after ~3 min of model download) — catch it now.

In [ ]:
import subprocess

try:
    result = subprocess.run(
        ['nvidia-smi',
         '--query-gpu=name,memory.total,memory.free,driver_version',
         '--format=csv,noheader'],
        capture_output=True, text=True, check=True, timeout=10,
    )
except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
    raise SystemExit(
        "❌ No GPU detected.\n\n"
        "Fix: Runtime → Change runtime type → Hardware accelerator → GPU.\n"
        "  - Free tier: pick T4 (16 GB VRAM — enough for 8B 4-bit LoRA)\n"
        "  - Colab Pro: pick L4 (22 GB — faster + more headroom)\n\n"
        "Save, restart the runtime, and re-run from cell 0.\n"
        f"(Underlying error: {type(e).__name__}: {e})"
    )

line = result.stdout.strip().splitlines()[0]
name, total_mib, free_mib, driver = [s.strip() for s in line.split(',')]
total_gb = int(total_mib.split()[0]) / 1024
free_gb = int(free_mib.split()[0]) / 1024

print(f'✅ GPU: {name}')
print(f'   VRAM total: {total_gb:.1f} GiB  (free: {free_gb:.1f} GiB)')
print(f'   Driver:     {driver}')

# 8B base + 4-bit + LoRA + bf16 activations + optimizer states needs
# roughly 13-14 GB at max_seq_length=4096. T4 (15 GB) just fits.
if total_gb < 14.0:
    print(f'\n⚠️  VRAM is tight ({total_gb:.1f} GiB). If you OOM during training:')
    print('    - lower max_seq_length to 3072 in cell 6')
    print('    - or drop gradient_accumulation_steps to 8')

## 1. Environment setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U \
    transformers==4.47.0 \
    peft==0.13.2 \
    'trl>=0.13,<0.16' \
    'bitsandbytes>=0.46' \
    'datasets>=3.6,<5' \
    accelerate==1.1.1
# Version notes:
#   datasets>=3.6 — avoids the dataclasses.fields() TypeError on Py3.12
#   bitsandbytes>=0.46 — has libbitsandbytes_cuda128.so for Colab's
#     CUDA 12.8 runtime AND fixes triton 3.x's removal of triton.ops
#   trl>=0.13 — SFTConfig.assistant_only_loss was added in 0.13

## 2. HF auth + load dataset from Drive

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Colab → 🔑 Secrets (left sidebar, key icon) → Add new secret
#   Name:               HF_TOKEN
#   Value:              <your token from huggingface.co/settings/tokens>
#   Notebook access:    toggle ON for this notebook
#
# Token permission: 'read' is enough for downloading the base model.
# Bump to 'write' only if you plan to push the adapter back to the Hub.
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    raise SystemExit(
        "❌ HF_TOKEN secret not found.\n"
        "Click the 🔑 icon in the left sidebar, add a secret named "
        "'HF_TOKEN', paste your Hugging Face token, and toggle "
        "'Notebook access' ON for this notebook. Then re-run this cell."
    )
except userdata.NotebookAccessError:
    raise SystemExit(
        "❌ HF_TOKEN secret exists but this notebook doesn't have access.\n"
        "Click the 🔑 icon in the left sidebar, find 'HF_TOKEN', and toggle "
        "'Notebook access' ON. Then re-run this cell."
    )

login(token=hf_token, add_to_git_credential=False)
print('✅ Hugging Face authenticated via Colab secret HF_TOKEN')

In [ ]:
import datasets as _ds_mod

# Guard against the datasets==3.1 + Python 3.12 TypeError.
# If you just upgraded datasets and see this error, RESTART THE RUNTIME
# (Runtime → Restart session) — pip upgrade doesn't reload cached modules.
_dv = tuple(int(x) for x in _ds_mod.__version__.split('.')[:2])
if _dv < (3, 6):
    raise SystemExit(
        f"❌ datasets=={_ds_mod.__version__} is too old — needs >=3.6.\n"
        "If you already upgraded: Runtime → Restart session, then re-run."
    )
print(f'datasets {_ds_mod.__version__} — OK')

# Extract dataset tar from Drive → local /content for fast access.
# We never mutate Drive during training (slow, network-bound).
!mkdir -p /content/dataset
!tar -xf "$KARIN_DATASET_TAR" -C /content/dataset/
!ls /content/dataset/

from datasets import load_from_disk
ds = load_from_disk('/content/dataset/sft_dataset.arrow')
print(ds)
print('\n--- SFT sample ---')
print({k: (v[:100] + '…' if isinstance(v, str) and len(v) > 100 else v) for k, v in ds['train_sft'][0]['messages'][1].items()})
print('--- assistant tool_calls ---')
print(ds['train_sft'][0]['messages'][-1])

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Defensive tool-call flattening (idempotent).
#
# mlabonne's abliterated Llama 3.1 chat template silently drops `tool_calls`
# on assistant messages and renders content=None as the literal string "None".
# If the tar on Drive was built before sft/scripts/build_dataset.py was
# patched (2026-04-19), its assistant turns still have tool_calls=[...] and
# content=None — training on it produces a model that outputs "None."
#
# This cell converts any surviving tool_calls into a plain JSON content that
# matches Karin's fallback parser (bridge/llm.py Shape B: name + arguments).
# No-op on already-flattened datasets.
# ──────────────────────────────────────────────────────────────────────
import json as _json

def _flatten_tool_calls(msgs):
    out = []
    for m in msgs:
        if m.get('role') == 'assistant' and m.get('tool_calls'):
            tc = m['tool_calls'][0]
            fn = tc.get('function') or {}
            name = fn.get('name') or tc.get('name', 'unknown')
            args = fn.get('arguments') or tc.get('arguments') or '{}'
            if isinstance(args, str):
                try: args = _json.loads(args)
                except Exception: pass
            rendered = _json.dumps({'name': name, 'arguments': args}, ensure_ascii=False)
            out.append({**m, 'content': rendered, 'tool_calls': None})
        else:
            out.append(m)
    return out

def _rewrite(ex):
    if 'messages' in ex:
        return {**ex, 'messages': _flatten_tool_calls(ex['messages'])}
    # DPO shape
    patched = dict(ex)
    for key in ('chosen', 'rejected'):
        if isinstance(ex.get(key), list):
            patched[key] = _flatten_tool_calls(ex[key])
    return patched

_before_sft = ds['train_sft'][0]['messages'][-1]
ds = ds.map(_rewrite)
_after_sft = ds['train_sft'][0]['messages'][-1]

_fixed = (_before_sft.get('tool_calls') and not _after_sft.get('tool_calls'))
print('tool-call flatten:', 'rewrote tool_calls -> content' if _fixed else 'already flat (no-op)')
print('sample after:', _after_sft.get('content', '')[:120])


## 3. Base model + tokenizer (4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# BASE_MODEL is defined in cell c_state (via CONFIG['base_model']).

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
print('Loaded', BASE_MODEL)

## 4. LoRA config

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

model = prepare_model_for_kbit_training(model)

if STATE['sft'] and not FORCE_REWRITE:
    # Resume path: attach the Drive-saved SFT adapter instead of training fresh.
    # Also mirror to /content so downstream cells (merge) find it locally.
    print(f"resume: loading SFT adapter from {RUN_DIR / 'adapter'}")
    model = PeftModel.from_pretrained(model, str(RUN_DIR / 'adapter'), is_trainable=True)
    _local_final = Path('/content/karin-sft/final')
    _local_final.parent.mkdir(parents=True, exist_ok=True)
    if _local_final.exists():
        shutil.rmtree(_local_final)
    shutil.copytree(RUN_DIR / 'adapter', _local_final)
else:
    lora_config = LoraConfig(
        r=CONFIG['lora_r'],
        lora_alpha=CONFIG['lora_alpha'],
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=CONFIG['lora_dropout'],
        bias='none',
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


## 5. SFT training

Writes checkpoints to local `/content/karin-sft` (fast SSD). We copy the final adapter to Drive at the end — never during training.

In [ ]:
if (STATE.get('sft') and not FORCE_REWRITE):
    print('skip: SFT training already done (sft=True)')
elif not CONFIG.get('run_sft', True):
    print("skip: SFT training disabled via CONFIG['run_sft']")
else:
    import trl
    print(f'trl version: {trl.__version__}')

    # --- config banner (pre-train sanity print) ---
    # Print the settings that most-often cause silent failures so a
    # Run-All user sees them before committing to training. If these
    # look wrong, abort before the 30-min SFT run commits to Drive.
    print('pre-train critical config:')
    print(f'  base_model            = {BASE_MODEL}')
    print(f'  dataset_version       = {CONFIG.get("dataset_version", "?")}')
    print(f'  sft_epochs            = {CONFIG["sft_epochs"]}')
    print(f'  max_seq_length        = {CONFIG["max_seq_length"]}')
    print(f'  lora r/alpha/dropout  = {CONFIG["lora_r"]}/{CONFIG["lora_alpha"]}/{CONFIG["lora_dropout"]}')
    print(f'  loss masking          = DataCollatorForCompletionOnlyLM # completion-only; required since iter-5')
    print(f'  packing               = False     # required by DataCollatorForCompletionOnlyLM')
    print(f'  seed                  = 42        # reproducibility')
    _sys = ds['train_sft'][0]['messages'][0]['content']
    print(f'  dataset system-prompt = {len(_sys)} chars, starts {_sys[:48]!r}')


    from trl import SFTConfig, SFTTrainer
    from transformers import EarlyStoppingCallback

    # ──────────────────────────────────────────────────────────────────────
    # Train/eval split + early stopping.
    #
    # EARLY_STOPPING = True carves 10% off train for an eval set, runs eval
    # every 5 steps, and stops when eval_loss hasn't improved for 3 evals.
    # Good insurance against overfitting — especially as the library grows
    # past ~500 examples. At 294 examples the eval split is small (~30) and
    # noisy, so it may stop prematurely on jitter. Set to False to disable
    # and train for the full 3 epochs.
    #
    # PATIENCE can be bumped to 5+ once you have more data, for a looser
    # stopping criterion.
    # ──────────────────────────────────────────────────────────────────────
    EARLY_STOPPING = True
    EVAL_FRACTION = CONFIG['eval_fraction']
    PATIENCE = 3
    EVAL_STEPS = 5

    if EARLY_STOPPING:
        _split = ds['train_sft'].train_test_split(test_size=EVAL_FRACTION, seed=42)
        train_sft = _split['train']
        eval_sft = _split['test']
        print(f'Train: {len(train_sft)}  Eval: {len(eval_sft)}')
    else:
        train_sft = ds['train_sft']
        eval_sft = None

    common_kwargs = dict(
        output_dir='/content/karin-sft',
        num_train_epochs=CONFIG['sft_epochs'],
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        # eval_accumulation_steps=1 forces logits to be moved off GPU after each
        # eval batch — without this, eval holds the full logits tensor for the
        # whole eval set and OOMs on A100 even with 40 GB.
        eval_accumulation_steps=1,
        # prediction_loss_only=True skips storing predictions during eval; the
        # eval_loss metric (what early stopping uses) still works.
        prediction_loss_only=True,
        learning_rate=CONFIG['sft_lr'],
        weight_decay=CONFIG['weight_decay'],
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        logging_steps=5,
        bf16=True,
        seed=42,                   # reproducibility
        # Was 4096 — eval OOM'd on A100 (logits = [1, 4096, 128256] fp32 = 2 GB,
        # cross_entropy intermediates push it past 8 GB per step). 3072 gives
        # ~950 tokens of headroom above the ~2100-token system prompt.
        # Drop to 2560 if 3072 still OOMs; bump back up if examples get longer.
        max_seq_length=CONFIG['max_seq_length'],
        packing=False,
        report_to='none',
    )


    # Completion-only loss via the TRL data collator. Version-
    # agnostic across trl>=0.7 (covers our 0.13-0.15 pin). The
    # collator finds the Llama-3.1 assistant-header token sequence
    # in each sample and sets labels=-100 for every position BEFORE
    # it — so the LoRA receives gradient signal ONLY on assistant
    # tokens. Equivalent in effect to SFTConfig(assistant_only_loss=
    # True) which TRL 0.15.2 no longer exposes. Requires packing=
    # False (already set) since packing would concatenate sequences
    # across sample boundaries and break the template-finder.
    from trl import DataCollatorForCompletionOnlyLM
    # Llama 3.1 multi-turn: pass BOTH instruction and response
    # templates so TRL can scan for alternating user/assistant
    # boundaries — more robust than relying on response_template
    # alone, which can fail to match due to standalone-vs-in-context
    # tokenization differences at the `\n\n` tail. Templates are
    # the HEADER PREFIX only (no trailing newlines); TRL encodes
    # them through the tokenizer and greps the token IDs in each
    # sample's input_ids. The header tokens stay consistent in
    # context, which is why this matches reliably.
    data_collator = DataCollatorForCompletionOnlyLM(
        instruction_template="<|start_header_id|>user<|end_header_id|>",
        response_template="<|start_header_id|>assistant<|end_header_id|>",
        tokenizer=tokenizer,
    )

    if EARLY_STOPPING:
        sft_config = SFTConfig(
            **common_kwargs,
            eval_strategy='steps',
            eval_steps=EVAL_STEPS,
            # save_strategy must match eval_strategy for load_best_model_at_end
            save_strategy='steps',
            save_steps=EVAL_STEPS,
            save_total_limit=5,
            load_best_model_at_end=True,
            metric_for_best_model='eval_loss',
            greater_is_better=False,
        )
        callbacks = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
    else:
        sft_config = SFTConfig(
            **common_kwargs,
            save_strategy='epoch',
            save_total_limit=3,
        )
        callbacks = []

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_sft,
        eval_dataset=eval_sft,
        data_collator=data_collator,
        processing_class=tokenizer,
        callbacks=callbacks,
    )

    # --- pre-train label-mask sanity probe (iter-5 readiness) ---
    # Verify DataCollatorForCompletionOnlyLM is actually masking
    # system/user/tool tokens before we commit GPU hours. On a
    # properly-masked chat-format batch, only ~5-15%% of tokens
    # should have non-ignore labels. If the ratio is near 100%%
    # something is wrong (response_template not found in the
    # tokenized sequence, dataset in wrong format, etc).
    import itertools as _it
    _probe = next(_it.islice(trainer.get_train_dataloader(), 1))
    _total = _probe['labels'].numel()
    _unmasked = (_probe['labels'] != -100).sum().item()
    _pct = 100.0 * _unmasked / max(_total, 1)
    print(f'loss-mask probe: {_unmasked}/{_total} tokens carry loss ({_pct:.1f}%). '
          f'Expect 0.5-5%% on this dataset shape (~8 KB system prompt + '
          f'short assistant turns). Validation run 2026-04-21 saw 1.0%%.')
    if _pct < 0.3 or _pct > 40.0:
        if _pct < 1.0:
            _diag = (
                f'UNDER-masked: {_pct:.2f}%% of tokens carry loss — collator '
                'is masking EVERYTHING. The instruction/response_template token '
                'IDs do not appear in the tokenized batch. Most likely cause: '
                'Llama 3.1 standalone-vs-in-context tokenization mismatch. '
                'Verify with:\n'
                "    tokenizer.encode('<|start_header_id|>assistant<|end_header_id|>', add_special_tokens=False)\n"
                'and check those IDs appear in trainer.get_train_dataloader() batches.'
            )
        else:
            _diag = (
                f'OVER-unmasked: {_pct:.1f}%% of tokens carry loss — the collator '
                'is leaking system/user tokens into the loss. Expected 5-15%% for '
                'chat-format SFT. Check template strings and packing=False.'
            )
        raise RuntimeError(f'loss-mask probe failed. {_diag}')
    # Decode the first handful of unmasked positions so the user can
    # eyeball that the model is learning on assistant text, not system text.
    _row0 = _probe['labels'][0]
    _positions = (_row0 != -100).nonzero(as_tuple=True)[0][:80]
    if len(_positions) > 0:
        _sample_ids = _probe['input_ids'][0][_positions]
        _sample = tokenizer.decode(_sample_ids, skip_special_tokens=False)
        print(f'sample learned text (row 0, first {len(_positions)} unmasked):')
        print(f'  {_sample[:300]!r}')

    trainer.train()
    trainer.save_model('/content/karin-sft/final')

    # Copy the adapter to Drive
    import shutil
    run_adapter = Path(os.environ['KARIN_RUN_DIR']) / 'adapter'
    if run_adapter.exists():
        shutil.rmtree(run_adapter)
    shutil.copytree('/content/karin-sft/final', run_adapter)
    print(f'✅ Adapter saved to {run_adapter}')

## 5b. Checkpoint picker — rank by routing accuracy, not eval_loss

`load_best_model_at_end=True` picks the lowest-loss checkpoint, but **eval_loss is a proxy** for what you actually care about (correct tool-call emission). Two checkpoints can tie on loss yet differ 10+ pp on routing accuracy.

This cell reloads each surviving checkpoint (set `save_total_limit=5` above), runs inference on ~40 held-out prompts from the training-split holdout, scores by tool-name match, and promotes the winner to `adapter/` on Drive. Adds a `checkpoint_picker.json` report so you can see the full ranking.

Takes ~15-20 min on T4 (5 checkpoints × 40 prompts × ~5 s each).

**⚠️ Caveat (learned iter-3).** The picker proxy uses a slice of the training split, which **anti-correlates with the real 135-case held-out eval after regularization** (r=8, dropout=0.1, wd=0.01). With iter-3's anti-overfit HPs, the picker tends to promote the *more-memorized* checkpoint — the one whose outputs match the training distribution best — while the less-memorized `load_best_model_at_end` pick generalizes better in production. Practical options:

  - **Skip this cell** when running iter-3 HPs and trust `eval_loss`.
  - **Replace the proxy** with a scaled-down curated routing holdout (~20-30 cases sampled from `sft/eval_cases_novel.yaml`) that mirrors production. Checkpoints ranked against that set DO correlate with the full 135-case eval. (Recommended long-term fix; see the "what's next" thread in `project_iter5_plan.md`.)
  - **Skip for a smoke run** — the `load_best_model_at_end` pick is fine for validating the pipeline.

Set `CONFIG['run_picker']=False` to skip entirely.


In [ ]:
# Post-training checkpoint picker — rank by routing accuracy, not eval_loss.
#
# eval_loss is a proxy; what we actually care about is "did the model emit
# the right tool call?". Two checkpoints can have near-identical loss but
# very different routing accuracy.
#
# Adapter-swap correctness: PeftModel.from_pretrained(base, adapter) injects
# LoraLayer wrappers INTO `base`. Calling it twice in a loop leaves stale
# injections around and every post-first checkpoint evaluates the SAME
# adapter (or the bare base). The fix is to wrap base once and use
# load_adapter + set_adapter to swap cleanly between named adapters.

import gc, re, json, random
from pathlib import Path
from peft import PeftModel

def _expected_tool(example):
    last = example['messages'][-1]
    # With the flattened dataset, tool calls live in content as JSON
    content = last.get('content') or ''
    m = re.search(r'"name"\s*:\s*"([a-z_][a-z0-9_]*)"', content)
    if m:
        return m.group(1)
    # Legacy path: still-structured tool_calls (if c3b didn't run)
    tc = last.get('tool_calls') or []
    if tc:
        return tc[0].get('function', {}).get('name') or tc[0].get('name', 'no_tool')
    return 'no_tool'

_TOOL_RE = re.compile(r'"name"\s*:\s*"([a-z_][a-z0-9_]*)"')

def _predicted_tool(text: str) -> str:
    m = _TOOL_RE.search(text or '')
    return m.group(1) if m else 'no_tool'

# Free the trainer's in-memory model — want a clean base
try:
    del model, trainer
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

if STATE['picker'] and not FORCE_REWRITE:
    print('skip: Checkpoint picker already done (picker=True)')
elif not CONFIG.get('run_picker', True):
    print("skip: Checkpoint picker disabled via CONFIG['run_picker']")
elif not any(Path('/content/karin-sft').glob('checkpoint-*')):
    print('skip: No local checkpoints (SFT was skipped this session)')
else:
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=torch.bfloat16,
    )
    base.eval()

    ckpt_root = Path('/content/karin-sft')
    ckpts = sorted(
        [p for p in ckpt_root.glob('checkpoint-*')
         if (p / 'adapter_config.json').exists()],
        key=lambda p: int(p.name.split('-')[1]),
    )
    if (ckpt_root / 'final' / 'adapter_config.json').exists():
        ckpts.append(ckpt_root / 'final')
    if not ckpts:
        raise SystemExit('No checkpoints found — did training complete?')

    # Load all adapters into a single PeftModel. The first one creates the
    # wrapper; the rest get registered as named adapters we swap via set_adapter.
    adapter_names = [f'ckpt_{i}' for i in range(len(ckpts))]
    m = PeftModel.from_pretrained(base, str(ckpts[0]), adapter_name=adapter_names[0])
    for name, ckpt in zip(adapter_names[1:], ckpts[1:]):
        m.load_adapter(str(ckpt), adapter_name=name)
    m.eval()

    # Proxy eval — up to 40 prompts from the held-out eval_sft split.
    N_PROBE = min(40, len(eval_sft))
    probe = random.Random(42).sample(list(eval_sft), N_PROBE)
    print(f'Scoring {len(ckpts)} checkpoints on {N_PROBE} held-out prompts...')

    scores = {}
    for ckpt, aname in zip(ckpts, adapter_names):
        m.set_adapter(aname)
        hits = 0
        for ex in probe:
            inp = tokenizer.apply_chat_template(
                ex['messages'][:-1],
                return_tensors='pt', add_generation_prompt=True,
            ).to(m.device)
            with torch.no_grad():
                out = m.generate(
                    inp, max_new_tokens=256,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    temperature=1.0, top_p=1.0,  # silence the sampling-param warnings
                )
            gen = tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True)
            if _predicted_tool(gen) == _expected_tool(ex):
                hits += 1
        scores[ckpt.name] = hits / N_PROBE
        print(f'  {ckpt.name}: {scores[ckpt.name]:.1%}')

    best_name = max(scores, key=scores.get)
    best_path = next(c for c in ckpts if c.name == best_name)
    best_aname = adapter_names[ckpts.index(best_path)]
    print(f'\nBest by routing accuracy: {best_name} @ {scores[best_name]:.1%}')
    print('(load_best_model_at_end picked: final, by eval_loss)')

    # Promote winner to Drive
    import shutil
    run_adapter = Path(os.environ['KARIN_RUN_DIR']) / 'adapter'
    if run_adapter.exists():
        shutil.rmtree(run_adapter)
    shutil.copytree(best_path, run_adapter)

    # Refresh /content/karin-sft/final so cell 7 (merge) picks this one
    finals = ckpt_root / 'final'
    if finals.exists() and best_path != finals:
        shutil.rmtree(finals)
        shutil.copytree(best_path, finals)

    picker_report = {
        'picked_checkpoint': best_name,
        'picked_routing_accuracy': scores[best_name],
        'all_scores': scores,
        'probe_size': N_PROBE,
        'selection_metric': 'tool_name_match',
        'note': 'Overrides load_best_model_at_end (which used eval_loss).',
    }
    (Path(os.environ['KARIN_RUN_DIR']) / 'checkpoint_picker.json').write_text(
        json.dumps(picker_report, indent=2)
    )
    print(f'Promoted to {run_adapter}')

    # ─── Self-verification: show what the winning adapter actually generates ───
    # Running this at the end of the picker means every run prints a
    # before/after you can eyeball. If the generated output doesn't look
    # like a JSON tool call, something upstream is broken.
    m.set_adapter(best_aname)
    _ex = next(x for x in probe if x['messages'][-1].get('content'))
    _train = tokenizer.apply_chat_template(_ex['messages'], tokenize=False)
    _inp = tokenizer.apply_chat_template(
        _ex['messages'][:-1], return_tensors='pt', add_generation_prompt=True,
    ).to(m.device)
    with torch.no_grad():
        _out = m.generate(
            _inp, max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            temperature=1.0, top_p=1.0,
        )
    _gen_clean = tokenizer.decode(_out[0][_inp.shape[1]:], skip_special_tokens=True)
    print('\n--- Debug: winner one-shot generation ---')
    print('Training target (last 200 chars):')
    print(_train[-200:])
    print('Generated :', repr(_gen_clean))
    print('Expected  :', repr(_ex['messages'][-1].get('content', '')))
    print(f'Match     : {_gen_clean.strip() == (_ex["messages"][-1].get("content") or "").strip()}')

    # Free picker state
    del m, base
    gc.collect(); torch.cuda.empty_cache()

# ── Reload model+winner so downstream cells (DPO/GRPO/merge) can continue.
from peft import PeftModel, prepare_model_for_kbit_training
print('\nReloading 4-bit base + winning adapter for downstream cells...')
_winner = Path(os.environ['KARIN_RUN_DIR']) / 'adapter'
_b = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config,
    device_map='auto', torch_dtype=torch.bfloat16,
)
_b.config.use_cache = False
_b = prepare_model_for_kbit_training(_b)
model = PeftModel.from_pretrained(_b, str(_winner), is_trainable=True)
model.print_trainable_parameters()


## 6. Optional DPO pass

Applies the preference pairs (decoys + ambiguity, plus iter-5's tool-output-usage and force-fire pairs). Teaches the model to prefer the no-tool response on negated / figurative prompts and to quote tool output rather than fall back on generic templates. **Skip for a smoke run** — come back after the first SFT eval shows the pipeline works.

**⚠️ Iter-4 post-mortem — read before kicking off iter-5 DPO.** iter-4's DPO was flattened to single-turn because TRL rejected multi-turn chats with `role: tool` as the last prompt message. The flatten workaround trained on a distribution that doesn't match multi-turn serve time and regressed tool-output usage 56.9% → 48.2% vs iter-3 (details in the intro's iter-4 post-mortem). **Iter-5 fixes this by passing multi-turn chat histories with `continue_final_message=True`** so TRL sees the same shape the bridge sends. Before training, verify `ds['train_dpo'][0]['chosen']` and `['rejected']` contain multi-turn message lists (not flat strings) — if they're flat, the dataset was built with the iter-4 shape and will repeat the regression.


In [ ]:
if (STATE.get('dpo') and not FORCE_REWRITE):
    print('skip: DPO training already done (dpo=True)')
elif not CONFIG.get('run_dpo', True):
    print("skip: DPO training disabled via CONFIG['run_dpo']")
else:
    from trl import DPOConfig, DPOTrainer

    dpo_config = DPOConfig(
        output_dir='/content/karin-dpo',
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=CONFIG['dpo_lr'],
        weight_decay=CONFIG['weight_decay'],
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        logging_steps=5,
        save_strategy='epoch',
        bf16=True,
        seed=42,                   # reproducibility
        max_length=4096,
        max_prompt_length=3072,
        beta=CONFIG['dpo_beta'],
        report_to='none',
    )

    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=dpo_config,
        train_dataset=ds['train_dpo'],
        processing_class=tokenizer,
    )

    dpo_trainer.train()
    dpo_trainer.save_model('/content/karin-dpo/final')

    # Copy to Drive
    run_adapter_dpo = Path(os.environ['KARIN_RUN_DIR']) / 'adapter_dpo'
    if run_adapter_dpo.exists():
        shutil.rmtree(run_adapter_dpo)
    shutil.copytree('/content/karin-dpo/final', run_adapter_dpo)
    print(f'✅ DPO adapter saved to {run_adapter_dpo}')

## 6b. Optional: GRPO instead of DPO

Replaces DPO with **Group Relative Policy Optimization** — online RL where the model samples multiple completions per prompt and a programmatic reward function ranks them. No preference pairs needed.

**When to use:** DPO has plateaued and the 40 preference pairs aren't enough to push routing higher. Routing is a perfect fit for GRPO because the reward is *verifiable* (regex-match the tool name) — no reward model needed.

**Alternative, not additive:** run EITHER DPO (cell 6) OR GRPO (this cell), not both. If you want GRPO, skip cell 6.

**Cost:** `num_generations=2` means 2× inference per prompt — roughly 5× DPO's wallclock. A 294-prompt epoch takes ~45-60 min on T4 / ~20 min on L4. Bump `num_generations` to 4 on L4+ for smoother gradients.

**Default:** `RUN_GRPO = False`. Flip to True to enable. Needs `trl>=0.14` (satisfied by cell 1's pin).


In [ ]:
# GRPO — alternative to DPO. Controlled by CONFIG['run_grpo'].
if STATE['grpo'] and not FORCE_REWRITE:
    print('skip: GRPO already done')
elif not CONFIG.get('run_grpo'):
    print("skip: GRPO disabled (CONFIG['run_grpo']=False)")
else:
    import re
    from trl import GRPOConfig, GRPOTrainer

    # Render each SFT example to a {prompt, expected_tool} pair.
    # GRPO samples completions from the current policy; we reward
    # completions whose parsed tool name matches expected_tool.
    def _to_grpo(ex):
        prompt = tokenizer.apply_chat_template(
            ex['messages'][:-1],
            tokenize=False, add_generation_prompt=True,
        )
        last = ex['messages'][-1]
        tcs = last.get('tool_calls') or []
        if tcs:
            fn = tcs[0].get('function') or {}
            expected = fn.get('name') or tcs[0].get('name', 'no_tool')
        else:
            expected = 'no_tool'
        return {'prompt': prompt, 'expected_tool': expected}

    grpo_ds = ds['train_sft'].map(
        _to_grpo, remove_columns=ds['train_sft'].column_names,
    )
    print(f'GRPO dataset: {len(grpo_ds)} prompts')
    print('sample:', {k: (v[:80] + '…' if isinstance(v, str) and len(v) > 80 else v)
                      for k, v in grpo_ds[0].items()})

    # Verifiable reward: tool-name match. Dense enough for 294 prompts;
    # consider partial credit (0.3 for any tool when a tool was expected)
    # if the model's reward stays flat at 0 after 50 steps.
    _TOOL_RE = re.compile(r'"name"\s*:\s*"([a-z_][a-z0-9_]*)"')

    def tool_match_reward(completions, expected_tool, **kwargs):
        rewards = []
        for comp, exp in zip(completions, expected_tool):
            text = comp if isinstance(comp, str) else comp[0]['content']
            m = _TOOL_RE.search(text or '')
            pred = m.group(1) if m else 'no_tool'
            rewards.append(1.0 if pred == exp else 0.0)
        return rewards

    grpo_config = GRPOConfig(
        output_dir='/content/karin-grpo',
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-6,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        logging_steps=5,
        save_strategy='epoch',
        bf16=True,
        # num_generations: rollouts per prompt. T4 (15 GB) fits 2 comfortably;
        # L4/A100 can handle 4-8 for smoother gradients. Each generation
        # adds ~500 MB KV cache at max_completion_length=512.
        num_generations=CONFIG['grpo_num_gens'],
        max_prompt_length=3072,
        max_completion_length=512,
        # beta: KL penalty toward the SFT reference policy.
        # 0.04 is TRL's default; lower → more exploration, higher → stay near SFT.
        beta=CONFIG['grpo_beta'],
        report_to='none',
    )

    grpo_trainer = GRPOTrainer(
        model=model,                            # SFT policy from cell 5
        reward_funcs=[tool_match_reward],
        args=grpo_config,
        train_dataset=grpo_ds,
        processing_class=tokenizer,
    )
    grpo_trainer.train()
    grpo_trainer.save_model('/content/karin-grpo/final')

    import shutil
    run_adapter_grpo = Path(os.environ['KARIN_RUN_DIR']) / 'adapter_grpo'
    if run_adapter_grpo.exists():
        shutil.rmtree(run_adapter_grpo)
    shutil.copytree('/content/karin-grpo/final', run_adapter_grpo)
    print(f'✅ GRPO adapter saved to {run_adapter_grpo}')


## 7. Merge LoRA → FP16 base

Picks DPO adapter if cell 6 ran, otherwise the SFT adapter. The merged FP16 model (~16 GB) stays on local `/content` — we never write it to Drive (too big). It's thrown away after GGUF conversion.

In [ ]:
if (STATE.get('merge_gguf') and not FORCE_REWRITE):
    print('skip: Merge (cell 8) already done (merge_gguf=True)')
elif not CONFIG.get('run_merge_gguf', True):
    print("skip: Merge (cell 8) disabled via CONFIG['run_merge_gguf']")
else:
    import gc
    for _n in ('model', 'trainer', 'dpo_trainer', 'grpo_trainer'):
        try:
            del globals()[_n]
        except KeyError:
            pass
    gc.collect(); torch.cuda.empty_cache()

    from peft import PeftModel

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )

    # Prefer GRPO > DPO > SFT. Check /content first (fresh session), Drive as fallback.
    _cands = (
        '/content/karin-grpo/final', str(RUN_DIR / 'adapter_grpo'),
        '/content/karin-dpo/final',  str(RUN_DIR / 'adapter_dpo'),
        '/content/karin-sft/final',  str(RUN_DIR / 'adapter'),
    )
    for _cand in _cands:
        if os.path.exists(_cand):
            ADAPTER = _cand
            break
    print('Merging adapter from:', ADAPTER)
    merged = PeftModel.from_pretrained(base, ADAPTER)
    merged = merged.merge_and_unload()
    merged.save_pretrained('/content/karin-merged', safe_serialization=True)
    tokenizer.save_pretrained('/content/karin-merged')
    print('Merged model written to /content/karin-merged')

## 8. Convert to GGUF → save to Drive

In [ ]:
if (STATE.get('merge_gguf') and not FORCE_REWRITE):
    print('skip: GGUF conversion (cell 8) already done (merge_gguf=True)')
elif not CONFIG.get('run_merge_gguf', True):
    print("skip: GGUF conversion (cell 8) disabled via CONFIG['run_merge_gguf']")
else:
    !git clone --depth=1 https://github.com/ggerganov/llama.cpp /content/llama.cpp

    # DO NOT `pip install -r requirements-convert_hf_to_gguf.txt`:
    # it pins torch~=2.2 which downgrades Colab's torch 2.10+cu128 to a
    # CPU-only wheel, breaks transformers' LlamaConfig import, and causes
    # convert_hf_to_gguf to silently fail after indexing the safetensors.
    # Install ONLY the gguf python module (and sentencepiece, used for
    # some tokenizers). All other deps — numpy, torch, transformers,
    # safetensors — are already satisfied by this notebook.
    !pip install -q gguf sentencepiece

    !python /content/llama.cpp/convert_hf_to_gguf.py \
        /content/karin-merged \
        --outfile /content/karin-fp16.gguf \
        --outtype f16

    !cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF && cmake --build build --target llama-quantize -j4

    !/content/llama.cpp/build/bin/llama-quantize \
        /content/karin-fp16.gguf \
        /content/karin-q4_k_m.gguf \
        Q4_K_M

    !ls -lh /content/karin-q4_k_m.gguf

    import shutil, json
    run_dir = Path(os.environ['KARIN_RUN_DIR'])
    shutil.copy2('/content/karin-q4_k_m.gguf', run_dir / 'karin-q4_k_m.gguf')

    config = {
        'base_model': CONFIG['base_model'],
        'lora_r': CONFIG['lora_r'],
        'lora_alpha': CONFIG['lora_alpha'],
        'epochs': CONFIG['sft_epochs'],
        'effective_batch_size': 16,
        'max_seq_length': CONFIG['max_seq_length'],
        'lr': CONFIG['sft_lr'],
        'dpo_ran': os.path.exists('/content/karin-dpo/final'),
        'n_sft_examples': len(ds['train_sft']) if 'ds' in dir() else None,
        'n_dpo_examples': len(ds['train_dpo']) if 'ds' in dir() else None,
        'quant': 'Q4_K_M',
    }
    with open(run_dir / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)

    print('\n' + '=' * 70)
    print(f'✅ Run artifacts saved to: {run_dir}')
    print('=' * 70)
    !ls -lh "$KARIN_RUN_DIR"

## 8b. Alternative: convert LoRA adapter directly to GGUF (no merge)

Skips the merge step entirely and converts the PEFT adapter to a GGUF adapter (~100-200 MB) that Ollama loads on top of the existing Jetson base model via an `ADAPTER` directive.

**Use this path when:**
- Cell 8 (`convert_hf_to_gguf.py`) failed with `LlamaConfig` import errors (pip downgraded torch to CPU-only)
- You want faster iteration — ship ~150 MB instead of ~5 GB
- You've already merged but want an adapter build too

**Caveat:** the Jetson base must match the training base weights. Training base = `mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated`, Jetson base = `mannix/llama3.1-8b-abliterated:tools-iq4_xs`. These are **different abliterations** — the LoRA directions may not align perfectly. Smoke test one prompt before the full 216-case eval; if quality is unexpectedly poor, fall back to the merge path.

In [ ]:
if (STATE.get('lora_gguf') and not FORCE_REWRITE):
    print("skip: LoRA GGUF (cell 8b) already done (lora_gguf=True)")
elif not CONFIG.get('run_lora_gguf', True):
    print("skip: LoRA GGUF (cell 8b) disabled via CONFIG['run_lora_gguf']")
else:
    # Convert the PEFT adapter directly to a GGUF adapter.
    # No HF model instantiation — sidesteps the torch/LlamaConfig downgrade bug.

    import os, shutil, json, subprocess
    from pathlib import Path

    run_dir = Path(os.environ['KARIN_RUN_DIR'])

    # Pick adapter: prefer GRPO > DPO > SFT (same order as merge path).
    candidates = [
        '/content/karin-grpo/final',
        '/content/karin-dpo/final',
        '/content/karin-sft/final',
        str(run_dir / 'adapter_grpo'),
        str(run_dir / 'adapter_dpo'),
        str(run_dir / 'adapter'),
    ]
    ADAPTER = next((p for p in candidates if os.path.exists(p)), None)
    if ADAPTER is None:
        raise SystemExit(
            'No adapter found. Run cell 5 (SFT) first, or point KARIN_RUN_DIR '
            'at a completed run in Drive.'
        )
    print(f'Adapter: {ADAPTER}')

    # Clone llama.cpp if not present
    if not os.path.exists('/content/llama.cpp'):
        os.system('git clone --depth=1 https://github.com/ggerganov/llama.cpp /content/llama.cpp')
    os.system('pip install -q gguf sentencepiece')

    # convert_lora_to_gguf.py wants --base to point at a LOCAL directory
    # containing config.json + tokenizer files. Snapshot-download resolves
    # the HF ID to a local cache path (uses the Drive-backed HF_HOME we set
    # up in cell 0b, so this is ~instant if the base is already cached).
    from huggingface_hub import snapshot_download
    print('Resolving base model snapshot path...')
    base_local = snapshot_download(
        CONFIG['base_model'],
        allow_patterns=['*.json', '*.txt', '*.md', 'tokenizer*', '*.safetensors*'],
    )
    print(f'Base local path: {base_local}')

    out_gguf = '/content/karin-lora.gguf'
    cmd = [
        'python', '/content/llama.cpp/convert_lora_to_gguf.py',
        '--base', base_local,
        '--outfile', out_gguf,
        '--outtype', 'f16',
        ADAPTER,
    ]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(out_gguf):
        print('--- STDOUT ---')
        print(result.stdout or '(empty)')
        print('--- STDERR ---')
        print(result.stderr or '(empty)')
        raise SystemExit(f'convert_lora_to_gguf failed (exit {result.returncode}).')
    print(result.stdout[-500:] if result.stdout else '(no stdout)')

    os.system(f'ls -lh {out_gguf}')

    # Ship to Drive
    drive_out = run_dir / 'karin-lora.gguf'
    shutil.copy2(out_gguf, drive_out)

    # Record what was built
    config_path = run_dir / 'config.json'
    config = json.loads(config_path.read_text()) if config_path.exists() else {}
    config.update({
        'base_model': CONFIG['base_model'],
        'artifact_kind': 'lora_adapter_gguf',
        'adapter_source': ADAPTER,
        'jetson_base_hint': 'mannix/llama3.1-8b-abliterated:tools-iq4_xs',
    })
    config_path.write_text(json.dumps(config, indent=2))

    print()
    print('=' * 70)
    print(f'LoRA adapter GGUF written to: {drive_out}')
    print('=' * 70)
    os.system(f'ls -lh "{run_dir}"')


## 9. Deploy on Jetson

**On your workstation**, pull the artifact from Drive. Three deploy options depending on what you built:

| Path | File | Size | Serving backend |
|------|------|------|-----------------|
| Cell 8 (merged GGUF) | `karin-q4_k_m.gguf` | ~5 GB | Ollama or llama.cpp |
| Cell 8b (LoRA GGUF)  | `karin-lora.gguf`   | ~150 MB | Ollama (`ADAPTER`) |

Local target (gitignored via `sft/runs/`):
```
sft/runs/<run_id>/
    ├── run_config.json          settings hash + hyperparams
    ├── config.json              build metadata
    ├── adapter/                 picker-promoted SFT adapter
    ├── adapter_dpo/             (if cell 6 ran)
    ├── adapter_grpo/            (if cell 6b ran)
    ├── karin-q4_k_m.gguf        merged build (if cell 8 ran)
    └── karin-lora.gguf          adapter build (if cell 8b ran)
```

`scp` to the Jetson (substitute your own SSH target):
```bash
scp sft/runs/<run_id>/karin-lora.gguf   <user>@<jetson-host>:~/karin_sft/
```

### Option A — Ollama, merged model

```bash
ollama show mannix/llama3.1-8b-abliterated:tools-iq4_xs --modelfile > Modelfile
sed -i 's|^FROM .*|FROM ./karin-q4_k_m.gguf|' Modelfile
ollama create karin-tuned -f Modelfile
```

### Option B — Ollama, LoRA adapter on top of existing mannix (current production path)

```bash
ollama show mannix/llama3.1-8b-abliterated:tools-iq4_xs --modelfile > Modelfile
echo 'ADAPTER ./karin-lora.gguf' >> Modelfile
ollama create karin-tuned -f Modelfile
```

This is how `karin-tuned:latest` (iter-3) is deployed today.

### Option C — llama.cpp (blocked on Orin Nano memory as of 2026-04-19)

llama.cpp looked like the migration target but ran out of memory during the 135-case eval: tool-schema injection pushes context above 4 KB, and raising `--ctx-size` exhausts the Orin Nano's 7.4 GiB usable unified memory when the prompt cache writes. Production stays on Ollama (Option B). Infra is kept under `deploy/systemd/karin-llama.service` + `deploy/systemd/karin.chat-template.jinja` for a future retry when the tool-schema footprint shrinks or the hardware upgrades.

### Smoke + eval

```bash
# Smoke:
ollama run karin-tuned 'what time is it'

# Held-out eval — `eval_cases_novel.yaml` is 135 cases (was 38 in iter-3):
cd ~/Karin/deploy && docker compose restart web
docker cp ~/Karin/scripts/eval_routing.py karin-web:/app/scripts/eval_routing.py
docker cp ~/karin_sft/eval_cases_novel.yaml karin-web:/app/scripts/eval_cases_novel.yaml
docker exec karin-web python3 /app/scripts/eval_routing.py \
    --cases /app/scripts/eval_cases_novel.yaml \
    --json /app/scripts/eval_$(date +%F)_karin_tuned.json
```

Per `feedback_eval_read_replies.md`, always manually read `final_reply` on flipped and failing cases before reporting a win — routing pass-rate + `used_tool_output` can mask real LoRA quality issues (eval stubs coincidentally sharing tokens with tool output fool the automated metric).

### Baselines to beat (on 135-case eval)

| Reference | Score |
|-----------|-------|
| Ollama + v4 mannix (no LoRA) | ~57% routing |
| Ollama + iter-3 LoRA alone | ~71% routing |
| Ollama + iter-3 + Phase-0 rescue (classifier + under-fire rescue) | **~85% routing (current production)** |
| Ollama + iter-4 LoRA + Phase-0 rescue | 74% routing; 48% tool-output usage (**ROLLED BACK**) |

**Iter-5 targets:** ≥90% routing and ≥75% tool-output usage. See `project_iter5_plan.md` for the dataset recipe.

The `sft/scripts/assert_disjoint.py` guard in `build_dataset.py` fails the build if a new training example verbatim-duplicates a test prompt. Don't bypass it — disjointness is what makes the eval number honest.


## 10. Release the Colab runtime

Free the GPU so you don't burn Pro credits while the notebook sits open.
This cell disconnects the runtime — any unsaved state in `/content/` is
lost, but all artifacts are already on Drive by this point.


In [ ]:
from google.colab import runtime
print('Terminating runtime...')
runtime.unassign()
